In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
class CustomCNN(Dataset):

    def __init__(self, root, train: bool, transform=None, download: bool = True):
        super().__init__()
        self.transform = datasets.ImageFolder(root=root, train=train, download=download)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.transform is not None:
            img = self.transform(img)
        return {"image": img, "label": int(label), "index": int(idx)}

In [ ]:
transform = transforms.Compose([
  transforms.Resize((28, 28))])

train_dataset = CustomCNN(root="./data", train=True, transform=transform, download=True)
test_dataset = CustomCNN(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)




In [ ]:
# Write your code here
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Load pretrained EfficientNetV2-Small
model = efficientnet_v2_s(weights="IMAGENET1K_V1")

# 1. Freeze the backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

# 2. Replace classifier head for 26 EMNIST letters
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 26)

model


In [ ]:

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = (labels - 1).to(device)   # shift labels 1–26 → 0–25

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = (labels - 1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

In [ ]:
# Write your code here

import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

In [ ]:
# Write your code here
